In [1]:
import torch
import pandas as pd
import os
import sys
sys.path.append('../')
import json

In [2]:
def count_parameters(path: str):
    model: dict = torch.load(os.path.join(path, "model.pth"), map_location='cpu', weights_only=True)
    num_params = 0
    num_params_non_zero = 0
    for k, v in model.items():
        num_params += v.numel()
        num_params_non_zero += (v != 0).sum().int().item()
    return int(num_params_non_zero)

In [3]:
models = [
    "/local/scratch/clmn1/videoNCA/CholecDataset/ethereal-surf-43",
    "/local/scratch/clmn1/videoNCA/CholecDataset/mild-microwave-36",
    "/local/scratch/clmn1/videoNCA/CholecDataset/rich-rain-27",
    "/local/scratch/clmn1/videoNCA/CholecDataset/unique-cherry-42",
    "/local/scratch/clmn1/videoNCA/CholecDataset/dashing-wildflower-38",

    "/local/scratch/clmn1/videoNCA/CataractsDataset/rural-bush-76",
    "/local/scratch/clmn1/videoNCA/CataractsDataset/electric-flower-74",
    "/local/scratch/clmn1/videoNCA/CataractsDataset/dashing-firefly-83",
    "/local/scratch/clmn1/videoNCA/CataractsDataset/breezy-sky-79",
    "/local/scratch/clmn1/videoNCA/CataractsDataset/charmed-mountain-80",
]
df = []
for model in models:
    dices = 100 * pd.read_csv(os.path.join(model, "dices_best_val.csv"))
    config = json.load(open(os.path.join(model, "config.json"), "r"))
    incremental_nca_params = config.get("incremental_nca_params", {
        "num_channels": 8,
        "hidden_size": 32,
    })
    num_channels = incremental_nca_params["num_channels"]
    hidden_size = incremental_nca_params["hidden_size"]
    df.append({
        "num_channels": num_channels,
        "hidden_size": hidden_size,
        "dataset": config["dataset"],
        "micro Dice": dices.stack().mean(),
        "micro Dice std": dices.stack().std(),
        "macro Dice": dices.mean(axis=0).mean(),
        "macro Dice std": dices.mean(axis=0).std(),
        "num_params": count_parameters(model),
    })
df = pd.DataFrame(df)
df["num_params"] = df["num_params"].astype(int)
df

,num_channels,hidden_size,dataset,micro Dice,micro Dice std,macro Dice,macro Dice std,num_params
0,8,32,CholecDataset,77.353944,20.845354,73.704735,11.667716,59625
1,8,16,CholecDataset,77.641096,21.366720,73.926559,12.008260,40185
2,8,8,CholecDataset,77.717237,20.666678,74.294480,11.923755,30465
3,4,16,CholecDataset,77.580461,20.968000,74.048861,11.977153,34785
4,4,8,CholecDataset,76.817681,21.586067,72.595870,13.142057,27465
5,8,32,CataractsDataset,81.665625,17.748472,75.860643,13.586867,148905
6,8,16,CataractsDataset,82.582823,17.556153,76.355778,12.675346,85625
7,8,8,CataractsDataset,81.608180,18.371307,75.232742,13.585498,53985
8,4,16,CataractsDataset,82.528811,17.619141,77.997866,9.651963,55785
9,4,8,CataractsDataset,82.135771,17.815775,75.596862,14.062138,42505


In [4]:
wide = df.pivot(
    index=["num_channels", "hidden_size"],
    columns="dataset",
    values=["micro Dice", "micro Dice std", "macro Dice", "macro Dice std", "num_params"]
).swaplevel(0, 1, axis=1).sort_index(axis=1)
desired_order = ["CholecDataset", "CataractsDataset"]

wide = wide.reindex(desired_order, axis=1, level=0)
wide = wide.sort_index(level=["num_channels", "hidden_size"], ascending=[False, False])

wide

dataset                  CholecDataset                            \
                            macro Dice macro Dice std micro Dice   
num_channels hidden_size                                           
8            32              73.704735      11.667716  77.353944   
             16              73.926559      12.008260  77.641096   
             8               74.294480      11.923755  77.717237   
4            16              74.048861      11.977153  77.580461   
             8               72.595870      13.142057  76.817681   

dataset                                            CataractsDataset  \
                         micro Dice std num_params       macro Dice   
num_channels hidden_size                                              
8            32               20.845354    59625.0        75.860643   
             16               21.366720    40185.0        76.355778   
             8                20.666678    30465.0        75.232742   
4            16               20.968000    34785.0        77.997866   
             8                21.586067    27465.0        75.596862   

dataset                                                                       
                         macro Dice std micro Dice micro Dice std num_params  
num_channels hidden_size                                                      
8            32               13.586867  81.665625      17.748472   148905.0  
             16               12.675346  82.582823      17.556153    85625.0  
             8                13.585498  81.608180      18.371307    53985.0  
4            16                9.651963  82.528811      17.619141    55785.0  
             8                14.062138  82.135771      17.815775    42505.0

In [5]:
wide_merged = wide.copy()

datasets = wide.columns.levels[0]

for ds in datasets:
    for metric in ["macro Dice", "micro Dice"]:
        mean_col = (ds, metric)
        std_col = (ds, metric + " std")

        if mean_col in wide.columns and std_col in wide.columns:
            wide_merged[(ds, metric)] = (
                wide[mean_col].map("{:.1f}".format)
                + " $\\pm$ "
                + wide[std_col].map("{:.1f}".format)
            )

            wide_merged = wide_merged.drop(columns=[std_col])

In [6]:
wide_merged

dataset                     CholecDataset                              \
                               macro Dice       micro Dice num_params   
num_channels hidden_size                                                
8            32           73.7 $\pm$ 11.7  77.4 $\pm$ 20.8    59625.0   
             16           73.9 $\pm$ 12.0  77.6 $\pm$ 21.4    40185.0   
             8            74.3 $\pm$ 11.9  77.7 $\pm$ 20.7    30465.0   
4            16           74.0 $\pm$ 12.0  77.6 $\pm$ 21.0    34785.0   
             8            72.6 $\pm$ 13.1  76.8 $\pm$ 21.6    27465.0   

dataset                  CataractsDataset                              
                               macro Dice       micro Dice num_params  
num_channels hidden_size                                               
8            32           75.9 $\pm$ 13.6  81.7 $\pm$ 17.7   148905.0  
             16           76.4 $\pm$ 12.7  82.6 $\pm$ 17.6    85625.0  
             8            75.2 $\pm$ 13.6  81.6 $\pm$ 18.4    53985.0  
4            16            78.0 $\pm$ 9.7  82.5 $\pm$ 17.6    55785.0  
             8            75.6 $\pm$ 14.1  82.1 $\pm$ 17.8    42505.0

In [7]:
formatters =({
    (dataset, "num_params"): lambda x: f"{int(x):,}"
    for dataset in wide.columns.levels[0]
})

df_latex = wide_merged.to_latex(
    formatters=formatters,
    multicolumn=True,
    multicolumn_format="c",
    escape=False,
)


print(df_latex)

\begin{tabular}{llllrllr}
\toprule
 & dataset & \multicolumn{3}{c}{CholecDataset} & \multicolumn{3}{c}{CataractsDataset} \\
 &  & macro Dice & micro Dice & num_params & macro Dice & micro Dice & num_params \\
num_channels & hidden_size &  &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{8} & 32 & 73.7 $\pm$ 11.7 & 77.4 $\pm$ 20.8 & 59,625 & 75.9 $\pm$ 13.6 & 81.7 $\pm$ 17.7 & 148,905 \\
 & 16 & 73.9 $\pm$ 12.0 & 77.6 $\pm$ 21.4 & 40,185 & 76.4 $\pm$ 12.7 & 82.6 $\pm$ 17.6 & 85,625 \\
 & 8 & 74.3 $\pm$ 11.9 & 77.7 $\pm$ 20.7 & 30,465 & 75.2 $\pm$ 13.6 & 81.6 $\pm$ 18.4 & 53,985 \\
\cline{1-8}
\multirow[t]{2}{*}{4} & 16 & 74.0 $\pm$ 12.0 & 77.6 $\pm$ 21.0 & 34,785 & 78.0 $\pm$ 9.7 & 82.5 $\pm$ 17.6 & 55,785 \\
 & 8 & 72.6 $\pm$ 13.1 & 76.8 $\pm$ 21.6 & 27,465 & 75.6 $\pm$ 14.1 & 82.1 $\pm$ 17.8 & 42,505 \\
\cline{1-8}
\bottomrule
\end{tabular}

